# EDA - The Movies Dataset

Descarga del dataset [`rounakbanik/the-movies-dataset`](https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset) via `kagglehub` y unificacion de sus CSVs (`movies_metadata.csv`, `credits.csv`, `keywords.csv`) en un unico objeto por pelicula.

Las columnas que en el CSV original vienen como JSON stringificado (generos, reparto, equipo tecnico, keywords, compañias/paises productores, idiomas) se parsean y se dejan como listas de valores (por ejemplo nombres), descartando los ids internos de esas entidades. El unico id que se conserva es el `id` de la pelicula, usado como referencia.

> Nota: `kagglehub` requiere credenciales de Kaggle configuradas (`~/.kaggle/kaggle.json` o las variables de entorno `KAGGLE_USERNAME` / `KAGGLE_KEY`).

In [1]:
import ast
import json
import os

import kagglehub
import numpy as np
import pandas as pd

c:\Users\Andres\proyectos\EPIALC\Catalogo-de-Peliculas\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Descarga del dataset

In [2]:
# Download latest version
path = kagglehub.dataset_download("rounakbanik/the-movies-dataset")

print("Path to dataset files:", path)
os.listdir(path)

Path to dataset files: C:\Users\Andres\.cache\kagglehub\datasets\rounakbanik\the-movies-dataset\versions\7


['credits.csv',
 'keywords.csv',
 'links.csv',
 'links_small.csv',
 'movies_metadata.csv',
 'ratings.csv',
 'ratings_small.csv']

## 2. Carga de los CSVs

Usamos `movies_metadata.csv`, `credits.csv` y `keywords.csv`. `low_memory=False` evita warnings por columnas con tipos mixtos (dataset conocido por tener algunas filas corruptas).

In [3]:
movies_metadata = pd.read_csv(os.path.join(path, "movies_metadata.csv"), low_memory=False)
credits = pd.read_csv(os.path.join(path, "credits.csv"))
keywords = pd.read_csv(os.path.join(path, "keywords.csv"))

movies_metadata.shape, credits.shape, keywords.shape

((45466, 24), (45476, 3), (46419, 2))

## 3. Limpieza de ids

`movies_metadata.csv` tiene algunas filas corruptas (columnas desalineadas) donde `id` no es numerico. Las descartamos.

In [4]:
movies_metadata["id"] = pd.to_numeric(movies_metadata["id"], errors="coerce")
movies_metadata = movies_metadata.dropna(subset=["id"]).copy()
movies_metadata["id"] = movies_metadata["id"].astype(int)

# nos quedamos con la primera aparicion de cada id
movies_metadata = movies_metadata.drop_duplicates(subset="id", keep="first")
credits = credits.drop_duplicates(subset="id", keep="first")
keywords = keywords.drop_duplicates(subset="id", keep="first")

movies_metadata.shape, credits.shape, keywords.shape

((45433, 24), (45432, 3), (45432, 2))

## 4. Parseo de columnas JSON stringificado

Columnas como `genres`, `production_companies`, `cast`, `crew`, `keywords` vienen como texto con formato tipo Python (comillas simples), no JSON valido. Se parsean con `ast.literal_eval` y se extraen solo los valores relevantes (ej. `name`), descartando los ids internos de cada entidad.

In [5]:
def safe_literal_eval(value):
    if pd.isna(value):
        return []
    try:
        return ast.literal_eval(value)
    except (ValueError, SyntaxError):
        return []


def extract_names(value, key="name"):
    items = safe_literal_eval(value)
    if not isinstance(items, list):
        return []
    return [item[key] for item in items if isinstance(item, dict) and key in item]


def extract_directors(crew_value):
    crew = safe_literal_eval(crew_value)
    if not isinstance(crew, list):
        return []
    return [member["name"] for member in crew if isinstance(member, dict) and member.get("job") == "Director"]


def extract_collection_name(value):
    if pd.isna(value):
        return None
    collection = safe_literal_eval(value)
    if isinstance(collection, dict):
        return collection.get("name")
    return None

In [6]:
movies_metadata["genres"] = movies_metadata["genres"].apply(extract_names)
movies_metadata["production_companies"] = movies_metadata["production_companies"].apply(extract_names)
movies_metadata["production_countries"] = movies_metadata["production_countries"].apply(extract_names)
movies_metadata["spoken_languages"] = movies_metadata["spoken_languages"].apply(extract_names)
movies_metadata["collection"] = movies_metadata["belongs_to_collection"].apply(extract_collection_name)

In [7]:
credits["cast"] = credits["cast"].apply(extract_names)
credits["directors"] = credits["crew"].apply(extract_directors)

In [8]:
keywords["keywords"] = keywords["keywords"].apply(extract_names)

## 5. Unificacion en un objeto por pelicula

Se combinan los tres datasets por `id` y se seleccionan los campos finales. Cada pelicula queda como un objeto con sus datos internos (generos, reparto, directores, keywords, etc.) como listas de valores, sin ids internos salvo el `id` de la pelicula.

In [9]:
metadata_cols = [
    "id", "title", "original_title", "overview", "tagline", "status",
    "release_date", "runtime", "budget", "revenue", "popularity",
    "vote_average", "vote_count", "original_language", "collection",
    "genres", "production_companies", "production_countries", "spoken_languages",
]

unified = (
    movies_metadata[metadata_cols]
    .merge(credits[["id", "cast", "directors"]], on="id", how="left")
    .merge(keywords[["id", "keywords"]], on="id", how="left")
)

for list_col in ["cast", "directors", "keywords"]:
    unified[list_col] = unified[list_col].apply(lambda v: v if isinstance(v, list) else [])

# pandas usa NaN/NaT para valores faltantes, que no son JSON valido (Python los
# serializa como el token literal `NaN`); los convertimos a None -> JSON null
unified = unified.astype(object).where(unified.notna(), None)

unified.shape

(45433, 22)

In [10]:
def clean_line_separators(value):
    # algunos textos (overview/tagline) traen caracteres Unicode Line/Paragraph
    # Separator, validos en JSON pero marcados por editores como VS Code como
    # "unusual line terminators"
    if isinstance(value, str):
        return value.replace(chr(0x2028), " ").replace(chr(0x2029), " ")
    if isinstance(value, list):
        return [clean_line_separators(v) for v in value]
    if isinstance(value, dict):
        return {k: clean_line_separators(v) for k, v in value.items()}
    return value


records = unified.to_dict(orient="records")
records = [clean_line_separators(r) for r in records]

print(json.dumps(records[0], indent=2, ensure_ascii=False, default=str))

{
  "id": 862,
  "title": "Toy Story",
  "original_title": "Toy Story",
  "overview": "Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences.",
  "tagline": null,
  "status": "Released",
  "release_date": "1995-10-30",
  "runtime": 81.0,
  "budget": "30000000",
  "revenue": 373554033.0,
  "popularity": "21.946943",
  "vote_average": 7.7,
  "vote_count": 5415.0,
  "original_language": "en",
  "collection": "Toy Story Collection",
  "genres": [
    "Animation",
    "Comedy",
    "Family"
  ],
  "production_companies": [
    "Pixar Animation Studios"
  ],
  "production_countries": [
    "United States of America"
  ],
  "spoken_languages": [
    "English"
  ],
  "cast": [
    "Tom Hanks",
    "Tim Allen",
    "Don Rickles",
    "Jim Varney",
  

## 6. Guardar el dataset unificado

In [11]:
output_dir = "data"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "movies_unified.json")

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2, default=str, allow_nan=False)

print(f"Guardadas {len(records)} peliculas en {output_path}")

Guardadas 45433 peliculas en data\movies_unified.json


In [12]:
sample_path = os.path.join(output_dir, "movies_unified_sample.json")

with open(sample_path, "w", encoding="utf-8") as f:
    json.dump(records[:10], f, ensure_ascii=False, indent=2, default=str, allow_nan=False)

print(f"Guardada muestra de 10 peliculas en {sample_path}")

Guardada muestra de 10 peliculas en data\movies_unified_sample.json


## 7. Muestra representativa: directores mas proliferos por genero

Se arma un subconjunto de `TAMANIO_MUESTRA` peliculas pensado para ser representativo
del dataset completo:

1. Se usa el primer genero de cada pelicula como "genero principal" (evita contar la
   misma pelicula en mas de un genero al calcular proporciones).
2. Se calcula la proporcion de cada genero en el dataset completo y se la traslada al
   tamaño de la muestra (muestreo estratificado), para que la distribucion de generos
   de la muestra se parezca a la original.
3. Dentro de cada genero, se identifican los `N_DIRECTORES_TOP` directores con mas
   peliculas en ese genero (los "mas proliferos") y se arma la muestra priorizando
   peliculas de esos directores.
4. Si las peliculas de esos directores no alcanzan para cubrir el cupo del genero, se
   completa con el resto de peliculas de ese genero, elegidas al azar.
5. Si aun asi falta para llegar a `TAMANIO_MUESTRA` (por redondeos), se completa al azar
   con el resto del dataset.

In [13]:
from collections import Counter

N_DIRECTORES_TOP = 5      # directores mas proliferos a considerar por genero
TAMANIO_MUESTRA = 1000    # cantidad total de peliculas en la muestra
SEMILLA_ALEATORIA = 42    # reproducibilidad de los sampleos al azar

# solo peliculas con al menos un genero informado
muestreo = unified[unified["genres"].apply(lambda g: len(g) > 0)].copy()
muestreo["genero_principal"] = muestreo["genres"].apply(lambda g: g[0])

# proporcion de cada genero en el dataset completo
conteo_generos = muestreo["genero_principal"].value_counts()
proporcion_generos = conteo_generos / conteo_generos.sum()

# cupo de peliculas por genero para que la muestra sea representativa
cupos_por_genero = (proporcion_generos * TAMANIO_MUESTRA).round().astype(int)

# el redondeo puede dejar la suma en 999 o 1001: se ajusta en el genero mas grande
diferencia = TAMANIO_MUESTRA - cupos_por_genero.sum()
if diferencia != 0:
    genero_mas_grande = cupos_por_genero.idxmax()
    cupos_por_genero[genero_mas_grande] += diferencia

cupos_por_genero.sort_values(ascending=False)

genero_principal
Drama              280
Comedy             205
Action             104
Documentary         79
Horror              61
Crime               39
Thriller            39
Adventure           35
Romance             28
Animation           26
Fantasy             16
Science Fiction     15
Mystery             13
Family              12
Music               11
Western             10
TV Movie             9
War                  9
History              6
Foreign              3
Name: count, dtype: int64

In [14]:
seleccionadas_ids: list[int] = []

for genero, cupo in cupos_por_genero.items():
    peliculas_genero = muestreo[muestreo["genero_principal"] == genero]

    # directores mas proliferos dentro de este genero (una pelicula puede tener varios directores)
    conteo_directores = Counter(
        director
        for lista_directores in peliculas_genero["directors"]
        for director in lista_directores
    )
    top_directores = {director for director, _ in conteo_directores.most_common(N_DIRECTORES_TOP)}

    candidatas = peliculas_genero[
        peliculas_genero["directors"].apply(lambda lista: any(d in top_directores for d in lista))
    ]

    if len(candidatas) >= cupo:
        elegidas = candidatas.sample(n=cupo, random_state=SEMILLA_ALEATORIA)
    else:
        faltantes = cupo - len(candidatas)
        resto_genero = peliculas_genero.drop(candidatas.index)
        relleno = resto_genero.sample(n=min(faltantes, len(resto_genero)), random_state=SEMILLA_ALEATORIA)
        elegidas = pd.concat([candidatas, relleno])

    seleccionadas_ids.extend(elegidas["id"].tolist())

# el redondeo por genero puede dejar la muestra corta: se completa al azar con el resto del dataset
seleccionadas_ids = list(dict.fromkeys(seleccionadas_ids))
if len(seleccionadas_ids) < TAMANIO_MUESTRA:
    faltan = TAMANIO_MUESTRA - len(seleccionadas_ids)
    resto_general = muestreo[~muestreo["id"].isin(seleccionadas_ids)]
    relleno_general = resto_general.sample(n=min(faltan, len(resto_general)), random_state=SEMILLA_ALEATORIA)
    seleccionadas_ids.extend(relleno_general["id"].tolist())

muestra_representativa = unified[unified["id"].isin(seleccionadas_ids)].copy()
muestra_representativa.shape

(1000, 22)

In [15]:
# verificacion: comparar la proporcion de cada genero en la muestra vs. el dataset original
proporcion_muestra = (
    muestra_representativa["genres"]
    .apply(lambda g: g[0] if len(g) > 0 else None)
    .value_counts(normalize=True)
)

comparacion_generos = pd.DataFrame({
    "proporcion_original": proporcion_generos,
    "proporcion_muestra": proporcion_muestra,
}).fillna(0)

comparacion_generos["diferencia_pp"] = (
    (comparacion_generos["proporcion_muestra"] - comparacion_generos["proporcion_original"]) * 100
)

comparacion_generos.sort_values("proporcion_original", ascending=False)

,proporcion_original,proporcion_muestra,diferencia_pp
Drama,0.278035,0.280,0.196506
Comedy,0.205066,0.205,-0.006618
Action,0.104371,0.104,-0.037068
Documentary,0.079389,0.079,-0.038871
Horror,0.060920,0.061,0.008027
Crime,0.039171,0.039,-0.017099
Thriller,0.038683,0.039,0.031749
Adventure,0.035100,0.035,-0.010037
Romance,0.027703,0.028,0.029653
Animation,0.026145,0.026,-0.014501


### Guardar la muestra representativa

In [16]:
registros_muestra_representativa = muestra_representativa.to_dict(orient="records")
registros_muestra_representativa = [clean_line_separators(r) for r in registros_muestra_representativa]

muestra_representativa_path = os.path.join(output_dir, "movies_sample_representativa.json")

with open(muestra_representativa_path, "w", encoding="utf-8") as f:
    json.dump(registros_muestra_representativa, f, ensure_ascii=False, indent=2, default=str, allow_nan=False)

print(f"Guardadas {len(registros_muestra_representativa)} peliculas en {muestra_representativa_path}")

Guardadas 1000 peliculas en data\movies_sample_representativa.json
